# 07 - Double machine learning

Every estimator in notebooks 01 and 02 relies on a model of the outcome, or of
treatment assignment, or of both. Those models are usually linear, and when the
truth is not linear the adjustment goes wrong — sometimes badly enough that
adjusting is worse than not adjusting at all.

Double machine learning replaces those models with flexible learners, and adds
two devices — orthogonalisation and cross-fitting — that keep the flexibility
from contaminating the treatment effect. This notebook shows a case where it
buys nothing, and a case where it is the difference between 4.1 and 2.0.

## Causal question

A programme is offered to individuals, and take-up depends on their
characteristics. What is the average effect of the programme on the outcome?

The question is the same as in notebook 02. What changes is how badly the
nuisance functions behave.

## Data and design

- **Unit of analysis:** one individual.
- **Treatment:** `treatment`, binary.
- **Outcome:** `outcome`, continuous.
- **Covariates:** `x1`, `x2`, `x3`, all measured pre-treatment.

We use two datasets deliberately.

**Dataset A** is the standard confounded generator from notebook 01. Its
propensity and outcome surfaces are close to linear.

**Dataset B** is purpose-built to break linear adjustment. Treatment probability
depends on `sin(2 * x1)` and `x2**2`, and the outcome depends on `cos(1.5 * x1)`
and `x2**2`. Nothing about it is exotic — smooth, ordinary non-linearity of the
kind real data has routinely — but a linear model cannot represent it.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from causal_inference_lab.data_generators import make_confounded_binary_treatment
from causal_inference_lab.diagnostics import overlap_summary
from causal_inference_lab.dml import double_machine_learning_ate
from causal_inference_lab.estimators import aipw_ate, difference_in_means, g_computation_ate, ipw_ate

COVARIATES = ["x1", "x2", "x3"]

dataset_a = make_confounded_binary_treatment(n=4_000, seed=11)
data_a = dataset_a.data


def make_nonlinear_confounding(n: int = 4_000, true_effect: float = 2.0, seed: int = 7):
    """Confounding that a linear model cannot represent."""
    rng = np.random.default_rng(seed)
    x1, x2, x3 = rng.normal(size=n), rng.normal(size=n), rng.normal(size=n)

    propensity = 1.0 / (1.0 + np.exp(-(1.4 * np.sin(2 * x1) + 1.1 * (x2**2 - 1.0) - 0.5 * x3)))
    treatment = rng.binomial(1, propensity)

    baseline = 3.0 * np.cos(1.5 * x1) + 2.0 * x2**2 + 0.8 * x3
    outcome = baseline + true_effect * treatment + rng.normal(scale=1.0, size=n)

    frame = pd.DataFrame(
        {"x1": x1, "x2": x2, "x3": x3, "treatment": treatment, "outcome": outcome}
    )
    return frame, propensity


data_b, true_propensity_b = make_nonlinear_confounding()
TRUE_EFFECT_B = 2.0

print(f"dataset A: n={len(data_a):,}, treated={data_a['treatment'].mean():.1%}, "
      f"true ATE={dataset_a.true_ate:.3f}")
print(f"dataset B: n={len(data_b):,}, treated={data_b['treatment'].mean():.1%}, "
      f"true ATE={TRUE_EFFECT_B:.3f}")

**Interpretation.** The two datasets look alike on the surface: same covariate
names, same treatment share, similar sample size. Nothing in a summary table
would warn you that one of them will break linear adjustment.

## Estimand

The **average treatment effect (ATE)**: the mean difference in outcomes if
everyone were treated versus if nobody were.

DML targets the same estimand as AIPW. It is a different route to it, not a
different quantity.

## Identification assumptions

1. **Conditional ignorability.** Given `x1`, `x2`, `x3`, treatment is as good as
   random. Untestable, and unchanged by any amount of machine learning — a
   flexible model of the wrong variables is still the wrong model.
2. **Overlap.** Every unit has a non-degenerate probability of either arm.
3. **Nuisance models converge fast enough.** Cross-fitting plus
   orthogonalisation means the ATE estimate tolerates nuisance error of order
   n^(-1/4) rather than requiring the n^(-1/2) that plug-in estimation needs.
4. **Cross-fitting is honest.** Each unit's nuisance predictions come from a
   model that never saw that unit.

Assumption 1 is the important one, and it is exactly the assumption DML does
*not* relax. What DML relaxes is the functional-form commitment in 3.

## Estimation

Start with dataset A, where the nuisance functions are nearly linear. If DML
were strictly better than AIPW, it would show here.

In [ ]:
def compare(frame: pd.DataFrame, truth: float, label: str) -> pd.DataFrame:
    """Run the estimator panel on one dataset."""
    rows = [
        ("naive difference in means", difference_in_means(frame).estimate),
        ("g-computation (linear)", g_computation_ate(frame, covariates=COVARIATES).estimate),
        ("IPW (logistic)", ipw_ate(frame, covariates=COVARIATES).estimate),
        ("AIPW (linear + logistic)", aipw_ate(frame, covariates=COVARIATES).estimate),
        (
            "DML (linear nuisances)",
            double_machine_learning_ate(frame, covariates=COVARIATES, n_splits=5, seed=11).estimate,
        ),
    ]
    table = pd.DataFrame(rows, columns=["estimator", "estimate"])
    table["error"] = table["estimate"] - truth
    print(f"{label}  (true effect {truth:.3f})")
    print(table.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    return table


table_a = compare(data_a, dataset_a.true_ate, "Dataset A - near-linear confounding")

**Interpretation.** Every adjusted estimator lands within about 0.08 of the
truth, and DML is indistinguishable from AIPW. On data this well behaved, the
extra machinery earns nothing. This is worth stating plainly, because the
temptation is to reach for the most sophisticated estimator by default: here it
would only cost you runtime and interpretability.

Now dataset B, with the same estimators and the same linear nuisance models.

In [ ]:
table_b = compare(data_b, TRUE_EFFECT_B, "Dataset B - non-linear confounding")

naive_error = abs(table_b.loc[0, "error"])
adjusted_error = abs(table_b.loc[3, "error"])
print(f"\nnaive error:            {naive_error:.3f}")
print(f"AIPW error:             {adjusted_error:.3f}")
print(f"adjustment made it {'worse' if adjusted_error > naive_error else 'better'} "
      f"by {abs(adjusted_error - naive_error):.3f}")

**Interpretation.** This is the result worth sitting with. The naive estimate is
3.74 against a truth of 2.00 — badly confounded, as expected. Every adjusted
estimator then moves *further away*, landing near 4.07. Adjusting made the
answer worse than doing nothing.

The mechanism is not subtle. The linear propensity model cannot represent
`sin(2 * x1)`, so its predicted probabilities are wrong in a structured way, and
the weights built from them reweight the sample towards the wrong units. The
linear outcome model cannot represent `x2**2`, so its residuals carry the very
confounding it was supposed to remove. AIPW is doubly robust, which means it is
consistent if *either* model is right — here both are wrong, and double
robustness offers nothing.

Note that nothing in the output announces this. All four adjusted estimators
agree closely with each other, which reads like corroboration and is in fact
just four methods sharing one bad assumption.

Now the same estimator with flexible nuisance models: gradient boosting for the
outcome, a random forest for treatment. Only the learners change.

In [ ]:
flexible = double_machine_learning_ate(
    data_b,
    covariates=COVARIATES,
    n_splits=5,
    seed=7,
    outcome_model=GradientBoostingRegressor(n_estimators=150, max_depth=3, random_state=7),
    treatment_model=RandomForestClassifier(n_estimators=150, min_samples_leaf=20, random_state=7),
)

print(f"DML with flexible nuisances: {flexible.estimate:.3f}")
print(f"true effect:                 {TRUE_EFFECT_B:.3f}")
print(f"error:                       {flexible.estimate - TRUE_EFFECT_B:+.3f}")
print()
print(f"for comparison, DML with linear nuisances: {table_b.loc[4, 'estimate']:.3f}")

**Interpretation.** 1.91 against a true 2.00 — an error of about 0.09, down from
2.08. The estimator, the estimand, the data and the identification assumptions
are all unchanged. The only difference is that the nuisance models can now
represent the shapes actually present in the data.

That is the entire case for DML, and it is narrower than it first appears: it
does not weaken the assumption that you measured the right covariates. It
removes the additional, usually unexamined assumption that their relationship to
treatment and outcome is linear.

## Diagnostics

The obvious question is whether anything short of knowing the truth would have
warned us. Two checks: the overlap implied by each propensity model, and how
well each model predicts treatment.

In [ ]:
linear_ps = (
    LogisticRegression(max_iter=1_000)
    .fit(data_b[COVARIATES], data_b["treatment"])
    .predict_proba(data_b[COVARIATES])[:, 1]
)
forest_ps = (
    RandomForestClassifier(n_estimators=150, min_samples_leaf=20, random_state=7)
    .fit(data_b[COVARIATES], data_b["treatment"])
    .predict_proba(data_b[COVARIATES])[:, 1]
)

summary = pd.DataFrame(
    {
        "linear model": overlap_summary(linear_ps),
        "forest model": overlap_summary(forest_ps),
        "true propensity": overlap_summary(true_propensity_b),
    }
)
print(summary.to_string(float_format=lambda v: f"{v:.3f}"))

print(f"\nunits with true propensity above 0.95: {(true_propensity_b > 0.95).mean():.1%}")
print(f"the linear model thinks that share is:  {(linear_ps > 0.95).mean():.1%}")
print(f"the forest model thinks that share is:  {(forest_ps > 0.95).mean():.1%}")

**Interpretation.** The linear model reports a comfortable, narrow propensity
distribution — it sees no overlap problem at all, because it cannot see the
structure that creates one. The forest recovers a spread much closer to the true
propensities, including the units near certainty of treatment.

So the diagnostic does carry a signal, but only by comparison. A linear
propensity model examined on its own looks reassuring precisely where it is
failing, which is the practical danger: the misspecification hides itself in the
diagnostic you would use to detect it.

## Uncertainty

Cross-fitting introduces a randomness of its own: the fold assignment. If the
estimate moves appreciably when only the split seed changes, the result is an
artefact of one partition rather than a property of the data.

In [ ]:
across_splits = []
for split_seed in (1, 2, 3):
    run = double_machine_learning_ate(
        data_b,
        covariates=COVARIATES,
        n_splits=5,
        seed=split_seed,
        outcome_model=GradientBoostingRegressor(n_estimators=150, max_depth=3, random_state=7),
        treatment_model=RandomForestClassifier(n_estimators=150, min_samples_leaf=20, random_state=7),
    )
    across_splits.append(run.estimate)
    print(f"split seed {split_seed}: {run.estimate:.3f}")

spread = max(across_splits) - min(across_splits)
print(f"\nspread across split seeds: {spread:.3f}")
print(f"distance from truth:       {abs(np.mean(across_splits) - TRUE_EFFECT_B):.3f}")

**Interpretation.** The estimates vary by roughly 0.02 across fold assignments,
an order of magnitude smaller than the distance from the truth, and two orders
smaller than the bias that misspecification produced. Fold randomness is not the
binding uncertainty here.

A full treatment would add a confidence interval. Bootstrapping DML means
refitting every nuisance model on every resample, which is expensive enough that
it is usually replaced by the influence-function variance the estimator can
produce analytically — not implemented in this project, and named in the
limitations below rather than quietly skipped.

## Limitations

- **DML does not weaken conditional ignorability.** An unmeasured confounder
  defeats it exactly as it defeats IPW. Flexibility in the nuisance models buys
  robustness to functional form and nothing else.
- **No analytic confidence interval is reported.** This project does not
  implement the influence-function standard error, and bootstrapping the
  flexible fit is impractical. The point estimates above are reported without
  intervals, which is a real gap.
- **Overlap is strained in dataset B.** About 6% of units have a true propensity
  above 0.95. DML tolerates that better than IPW does, but no estimator
  manufactures information about units that are nearly always treated.
- **The learners are unturned defaults.** Depth, tree count, and leaf size were
  chosen by hand. In practice these would be cross-validated, and the choice
  interacts with the n^(-1/4) rate requirement.
- **Dataset B is constructed.** Its non-linearity was designed to defeat linear
  adjustment. That makes the demonstration clean, and it also means the size of
  the failure is a property of this construction, not a general magnitude.
- **Five folds is a default.** Fold count trades bias against the variance of
  each nuisance fit; it was not tuned.